# Vani-Kanoon Legal LLM Training

Fine-tune a small LLM for Indian legal assistance using LoRA.

## Setup Instructions:
1. Upload `training_data.json` to Kaggle
2. Enable GPU (Settings → Accelerator → GPU T4 x2 or P100)
3. Enable Internet (Settings → Internet → On)
4. Run all cells

In [ ]:
# Install required packages
!pip install -q transformers datasets accelerate peft bitsandbytes trl wandb

In [ ]:
import json
import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Load Training Data

In [ ]:
# Load your training data
# Option 1: If uploaded as Kaggle dataset
# data_path = "/kaggle/input/vani-kanoon-training/training_data.json"

# Option 2: If uploaded directly to notebook
data_path = "training_data.json"

with open(data_path, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

print(f"Loaded {len(raw_data)} training examples")

In [ ]:
# Format data for instruction tuning
def format_instruction(example):
    """Format each example into instruction-following format"""
    instruction = example.get('instruction', '')
    input_text = example.get('input', '')
    output = example.get('output', '')
    
    if input_text:
        text = f"""### Instruction:
{instruction}

### Input:
{input_text}

### Response:
{output}"""
    else:
        text = f"""### Instruction:
{instruction}

### Response:
{output}"""
    
    return {"text": text}

# Create dataset
formatted_data = [format_instruction(ex) for ex in raw_data]
dataset = Dataset.from_list(formatted_data)

print(f"Dataset size: {len(dataset)}")
print(f"\nSample entry:\n{dataset[0]['text'][:500]}...")

## 2. Load Base Model

We'll use **Mistral-7B-Instruct** as the base model. It's:
- Small enough to fine-tune on Kaggle GPU
- Good at following instructions
- Supports multiple languages

In [ ]:
# Model configuration
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"
# Alternative: "meta-llama/Llama-2-7b-chat-hf" (needs HF token)
# Alternative: "google/gemma-7b-it" (smaller, faster)

# Quantization config for 4-bit loading (saves memory)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading model: {MODEL_NAME}")

In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Tokenizer loaded. Vocab size: {tokenizer.vocab_size}")

In [ ]:
# Load model with quantization
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Prepare for training
model = prepare_model_for_kbit_training(model)

print(f"Model loaded successfully!")
print(f"Model parameters: {model.num_parameters():,}")

## 3. Configure LoRA

LoRA (Low-Rank Adaptation) lets us fine-tune only a small subset of parameters, making training fast and memory-efficient.

In [ ]:
# LoRA configuration
lora_config = LoraConfig(
    r=16,                      # Rank of the update matrices
    lora_alpha=32,             # Scaling factor
    target_modules=[           # Which layers to fine-tune
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# Apply LoRA to model
model = get_peft_model(model, lora_config)

# Print trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")

## 4. Training

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir="./vani-kanoon-legal-llm",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    max_grad_norm=0.3,
    lr_scheduler_type="cosine",
    report_to="none",  # Set to "wandb" if you want logging
)

print("Training configuration ready!")

In [ ]:
# Initialize trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    tokenizer=tokenizer,
    args=training_args,
    dataset_text_field="text",
    max_seq_length=2048,
    packing=False,
)

print("Trainer initialized!")

In [ ]:
# Start training
print("Starting training...")
print("="*50)

trainer.train()

print("="*50)
print("Training completed!")

## 5. Save the Model

In [ ]:
# Save the LoRA adapter
ADAPTER_PATH = "./vani-kanoon-lora-adapter"
model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)

print(f"LoRA adapter saved to: {ADAPTER_PATH}")

In [ ]:
# Merge LoRA with base model and save full model
from peft import AutoPeftModelForCausalLM

# Reload in full precision for merging
merged_model = AutoPeftModelForCausalLM.from_pretrained(
    ADAPTER_PATH,
    device_map="auto",
    torch_dtype=torch.float16,
)

# Merge and unload
merged_model = merged_model.merge_and_unload()

# Save merged model
MERGED_PATH = "./vani-kanoon-merged"
merged_model.save_pretrained(MERGED_PATH)
tokenizer.save_pretrained(MERGED_PATH)

print(f"Merged model saved to: {MERGED_PATH}")

## 6. Test the Model

In [ ]:
# Test the fine-tuned model
def generate_response(prompt, max_length=512):
    formatted_prompt = f"""### Instruction:
{prompt}

### Response:
"""
    
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")
    
    outputs = merged_model.generate(
        **inputs,
        max_new_tokens=max_length,
        temperature=0.7,
        do_sample=True,
        top_p=0.9,
        repetition_penalty=1.1,
    )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract just the response part
    response = response.split("### Response:")[-1].strip()
    return response

In [ ]:
# Test with legal questions
test_questions = [
    "What is the punishment for murder under BNS 2023?",
    "How to file an FIR in India?",
    "What are the grounds for divorce in India?",
    "धारा 302 में क्या प्रावधान है?",
]

for q in test_questions:
    print(f"\n{'='*60}")
    print(f"Question: {q}")
    print(f"{'='*60}")
    response = generate_response(q)
    print(f"Answer: {response[:1000]}..." if len(response) > 1000 else f"Answer: {response}")

## 7. Export for Download

Create a zip file to download the model.

In [ ]:
import shutil

# Create zip of the LoRA adapter (smaller, recommended)
shutil.make_archive("vani-kanoon-lora", 'zip', ADAPTER_PATH)
print("Created: vani-kanoon-lora.zip")

# Check file size
import os
size_mb = os.path.getsize("vani-kanoon-lora.zip") / (1024 * 1024)
print(f"LoRA adapter size: {size_mb:.2f} MB")

## 8. Upload to Hugging Face (Optional)

You can upload your model to Hugging Face for easy access.

In [ ]:
# Uncomment to upload to Hugging Face
# from huggingface_hub import login
# login(token="YOUR_HF_TOKEN")

# # Push to Hub
# merged_model.push_to_hub("YOUR_USERNAME/vani-kanoon-legal-llm")
# tokenizer.push_to_hub("YOUR_USERNAME/vani-kanoon-legal-llm")

## Done!

### Next Steps:
1. Download `vani-kanoon-lora.zip` from the output
2. For offline mobile use:
   - Convert to GGUF format using `llama.cpp`
   - Or use the model with Ollama
3. Integrate into your app's `offlineLegalLLM.js`

### To use with Ollama:
```bash
# Create Modelfile
echo 'FROM ./vani-kanoon-merged
PARAMETER temperature 0.7
SYSTEM "You are Vani-Kanoon, an expert Indian legal assistant."' > Modelfile

# Create Ollama model
ollama create vani-kanoon -f Modelfile

# Run
ollama run vani-kanoon
```